In [34]:
import scipy as sp
import pandas as pd
import numpy as np

In [ ]:
genotypes = pd.read_excel('preprocess_sub/genotypes_phen_preprocessed.xlsx')

In [ ]:
# save the information about each snp
snp_details = genotypes.iloc[:, [0, 1, 2]]
snp_details

In [37]:
genotypes.set_index(genotypes.columns[0], inplace=True)

In [38]:
phenotypes = pd.read_excel('preprocess_sub/phenotypes_preprocessed.xlsx')
phenotypes.set_index(phenotypes.columns[0], inplace=True)
phe_indexes = phenotypes.index.tolist()
phenotypes

In [40]:
genotypes

,Chr_Build37,Build37_position,BXD1,BXD2,BXD5,BXD6,BXD8,BXD9,BXD11,BXD12,...,BXD94,BXD95,BXD96,BXD97,BXD98,BXD99,BXD100,BXD101,BXD102,BXD103
Locus,,,,,,,,,,,,,,,,,,,,,
rs6269442,1,3482276,B,B,D,D,D,B,B,D,...,D,D,B,B,B,B,B,U,U,U
rs6376963,1,5008090,B,B,D,D,D,B,B,D,...,D,D,B,B,B,B,B,U,U,U
rs3677817,1,5176059,B,B,D,D,D,B,B,D,...,D,D,B,B,B,B,B,U,U,U
rs8236463,1,5579194,B,B,D,D,D,B,D,D,...,D,D,B,B,B,B,B,U,U,U
rs6298633,1,6820242,B,B,D,D,D,B,D,D,...,D,D,B,B,B,B,B,U,U,U
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
gnfX.141.820,20,158023554,B,B,B,B,B,B,B,B,...,D,H,B,B,D,D,B,U,U,U
CEL-X_154048891,20,160516970,B,B,B,B,B,B,B,B,...,D,H,B,B,D,D,B,U,U,U
CEL-X_154259201,20,160727280,B,B,B,D,B,B,B,B,...,D,H,B,B,D,D,B,U,U,U


In [42]:
# clean my phenotype data from null values
phenotypes = phenotypes.dropna()
# clean genotype data from unknown alleles
genotypes = genotypes.dropna()

In [43]:
filtered_phenotype = phenotypes.iloc[:, 6:]
filtered_genotype = genotypes.iloc[:, 2:]

In [44]:
filtered_genotype

,BXD1,BXD2,BXD5,BXD6,BXD8,BXD9,BXD11,BXD12,BXD13,BXD14,...,BXD94,BXD95,BXD96,BXD97,BXD98,BXD99,BXD100,BXD101,BXD102,BXD103
Locus,,,,,,,,,,,,,,,,,,,,,
rs6269442,B,B,D,D,D,B,B,D,B,B,...,D,D,B,B,B,B,B,U,U,U
rs6376963,B,B,D,D,D,B,B,D,B,B,...,D,D,B,B,B,B,B,U,U,U
rs3677817,B,B,D,D,D,B,B,D,B,B,...,D,D,B,B,B,B,B,U,U,U
rs8236463,B,B,D,D,D,B,D,D,B,B,...,D,D,B,B,B,B,B,U,U,U
rs6298633,B,B,D,D,D,B,D,D,B,B,...,D,D,B,B,B,B,B,U,U,U
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
gnfX.141.820,B,B,B,B,B,B,B,B,D,B,...,D,H,B,B,D,D,B,U,U,U
CEL-X_154048891,B,B,B,B,B,B,B,B,D,B,...,D,H,B,B,D,D,B,U,U,U
CEL-X_154259201,B,B,B,D,B,B,B,B,D,B,...,D,H,B,B,D,D,B,U,U,U


In [45]:
filtered_genotype, filtered_phenotype = filtered_genotype.align(filtered_phenotype, join='inner', axis=1)

In [46]:
filtered_genotype = filtered_genotype.applymap(lambda strain: 0 if strain=='B' else (2 if strain == 'D' else 1)) # change to numeric values

In [47]:
# Statistical F test
def f_test(r_square, n, k):
    dfn = n - 2
    dfd = k - 1
    f_statistics = r_square / ((1 - r_square) / (n - 2))
    p_val = 1 - sp.stats.f.cdf(f_statistics, dfd, dfn)
    return p_val

In [48]:
# Linear regression function
def linear_reg(x, y):
    k = 2
    n = x.shape[0]
    mean_x = x.mean()
    mean_y = y.mean()
    var_x = x.var()
    cov_x_y = (sum(x * y) - (n * mean_x * mean_y)) / (n - 1)
    b1 = cov_x_y / var_x
    b0 = mean_y - (b1 * mean_x)
    y_hat = b0 + b1 * x
    sse = sum(np.square(y - y_hat))
    ssr = sum((np.square(y_hat - mean_y)))
    sst = sse + ssr
    r_square = ssr / sst
    p_val = f_test(r_square, n, k)
    return b0, b1, p_val

<a id="Calculate p-values"></a>
# Calculate p-values

In [49]:
p_vals_dict = dict()
for phe in filtered_phenotype.iterrows():
    p_vals_dict[phe[0]] = dict()
    for snp, snp_info in zip(filtered_genotype.iterrows(),snp_details.iterrows()):
        p_val = linear_reg(x=snp[1].to_numpy(), y=phe[1].to_numpy())[2]
        p_vals_dict[phe[0]][snp_info[1]['Locus']] = [p_val,snp_info[1]['Chr_Build37']]

In [50]:
p_vals_dfs = dict()
for phe in phe_indexes:
    p_vals_dfs[phe] = pd.DataFrame.from_dict(p_vals_dict[phe], orient='index')
    p_vals_dfs[phe].rename(columns={0: 'p-value', 1: 'Chr'}, inplace=True)

<b> Manhattan Plot

In [59]:
from bioinfokit import analys, visuz 
p_val_to_plot = p_vals_dfs[phe_indexes[0]].copy()
visuz.marker.mhat(df=p_vals_dfs[phe_indexes[0]], chr='Chr',pv='p-value', gwasp=True)

In [60]:
p_val_to_plot = p_vals_dfs[phe_indexes[1]].copy()
visuz.marker.mhat(df=p_vals_dfs[phe_indexes[1]], chr='Chr',pv='p-value', gwasp=True)

In [63]:
p_vals_output = dict()
for phe in phe_indexes:
    corrected_alpha = 0.05/(p_vals_dfs[phe].shape[0]*2)
    p_vals_output[phe] = p_vals_dfs[phe][p_vals_dfs[phe]['p-value'] <= corrected_alpha]
print(p_vals_output)

{1003:             p-value  Chr     tpval
rs3721803  0.000007   10  5.151646
rs3695003  0.000007   10  5.151646
D10Mit28   0.000006   10  5.211049, 1002: Empty DataFrame
Columns: [p-value, Chr, tpval]
Index: []}


In [64]:
p_vals_output[phe_indexes[1]]

,p-value,Chr,tpval


<a id="Export"></a>
# Export

In [65]:
for phe_index in phe_indexes:
    (p_vals_dfs[phe_index].drop(columns=['p-value']).rename(columns={'tpval': '-log10(p-value)'})).to_excel('p-vals_'+str(phe_index)+'.xlsx')
    (p_vals_output[phe_index].drop(columns=['p-value']).rename(columns={'tpval': '-log10(p-value)'})).to_excel('QTLs_'+str(phe_index)+'.xlsx')